# Competition 1: Text Feature Engineering Report

**Student ID:** 114062516  
**Name:** 李子賢

In [1]:
import warnings
warnings.filterwarnings("ignore")

%matplotlib inline
import pandas as pd
import numpy as np
import re
from bs4 import BeautifulSoup

from sklearn.model_selection import KFold, GridSearchCV, cross_val_score
from sklearn.preprocessing import OneHotEncoder
from sklearn.feature_extraction.text import CountVectorizer
from scipy.sparse import hstack, csr_matrix

from category_encoders import TargetEncoder
import lightgbm as lgb

import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# === NLTK Setup ===
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

stop = stopwords.words('english')
lemmatizer = WordNetLemmatizer()

[nltk_data] Downloading package stopwords to
[nltk_data]     /home/kevin110062222/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     /home/kevin110062222/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     /home/kevin110062222/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


## Load Dataset

In [2]:
train_path = './datasets/train.csv'
test_path = './datasets/test.csv'

train_raw = pd.read_csv(train_path)
test_raw = pd.read_csv(test_path)

train_raw.head()

,Id,Popularity,Page content
0,0,-1,"<html><head><div class=""article-info""> <span c..."
1,1,1,"<html><head><div class=""article-info""><span cl..."
2,2,1,"<html><head><div class=""article-info""><span cl..."
3,3,-1,"<html><head><div class=""article-info""><span cl..."
4,4,-1,"<html><head><div class=""article-info""><span cl..."


## Basic Label Preparation

In [3]:
train = train_raw.copy()
test = test_raw.copy()

y = (train['Popularity'] == 1).astype(int)
train['Popularity_binary'] = y

print(train['Popularity_binary'].value_counts())


Popularity_binary
0    14011
1    13632
Name: count, dtype: int64


## 1. Data Preprocessing

### 1.1 Cleaning and Parsing
- Converted text to lowercase, removed non-alphanumeric characters using regex, and trimmed whitespace.

In [4]:
def preprocessor(text):
    soup = BeautifulSoup(text, "html.parser")
    for s in soup(["script", "style"]):
        s.decompose()
    footer = soup.find("footer", {"class": "article-topics"})
    if footer:
        footer.decompose()
    text = soup.get_text(separator=" ")
    text = re.sub(r"[\W_]+", " ", text.lower())
    return text.strip()


### 1.2 Tokenization and Lemmatization

- Split text via regex-based whitespace splitting.

- Used **WordNet Lemmatizer** and removed `NLTK English stopwords`.

In [5]:
def tokenizer(text):
    return re.split(r'\s+', text.strip())

def tokenizer_lemma_nostop(text):
    return [
        lemmatizer.lemmatize(w)
        for w in re.split(r"\s+", text.strip())
        if w not in stop and re.match('[a-zA-Z]+', w)
    ]

### 1.3 Feature Engineering
### (a) Channel Extraction

- Extracted `data-channel` attribute from `<article>` tags.

- Applied K-Fold Target Encoding (5 folds, smoothing = 0.3) to encode the channel feature, avoiding target leakage.

In [6]:
def extract_channel(html):
    soup = BeautifulSoup(html, 'html.parser')
    article = soup.find("article")
    return article.get("data-channel") if article else None

train['channel'] = train['Page content'].apply(extract_channel)
test['channel'] = test['Page content'].apply(extract_channel)

kf = KFold(n_splits=5, shuffle=True, random_state=42)
channel_te = np.zeros(len(train))

for tr_idx, va_idx in kf.split(train):
    te = TargetEncoder(cols=['channel'], smoothing=0.3)
    te.fit(train.iloc[tr_idx]['channel'], y.iloc[tr_idx])
    channel_te[va_idx] = te.transform(train.iloc[va_idx]['channel']).values.ravel()

train['channel_te'] = channel_te

te_final = TargetEncoder(cols=['channel'])
te_final.fit(train['channel'], y)
test['channel_te'] = te_final.transform(test['channel']).values.ravel()

### (b) Time Extraction

- Parsed `<time datetime="...">` tags and converted to datetime.

- Derived:
    - `weekday`, `hour`, `days_since_start`

In [7]:
def extract_time(html):
    soup = BeautifulSoup(html, 'html.parser')
    time_tag = soup.find("time")
    if time_tag and time_tag.get("datetime"):
        return pd.to_datetime(time_tag.get("datetime"), errors="coerce")
    return pd.NaT

train['time'] = train['Page content'].apply(extract_time)
test['time'] = test['Page content'].apply(extract_time)
train['time'] = pd.to_datetime(train['time'], errors='coerce')
test['time'] = pd.to_datetime(test['time'], errors='coerce')

train['weekday'] = train['time'].dt.weekday.fillna(0).astype(int)
test['weekday'] = test['time'].dt.weekday.fillna(0).astype(int)

train['hour'] = train['time'].dt.hour.fillna(0).astype(int)
test['hour'] = test['time'].dt.hour.fillna(0).astype(int)

ref_date = train['time'].min()
train['days_since_start'] = (train['time'] - ref_date).dt.days.fillna(0).astype(int)
test['days_since_start'] = (test['time'] - ref_date).dt.days.fillna(0).astype(int)

### (c) Topic Extraction

- Parsed footer links `(<footer class="article-topics">)`, cleaned with regex.

- Transformed using CountVectorizer with lemmatization and stopword removal (min_df=2).

In [8]:
def extract_topics(html):
    soup = BeautifulSoup(html, 'html.parser')
    return [a.get_text(strip=True) for a in soup.select("footer.article-topics a")]

def clean_topics(topics):
    if not topics:
        return ""
    text = " ".join(topics).lower()
    text = re.sub(r"[^a-z0-9\.]+", " ", text)
    return re.sub(r"\s+", " ", text).strip()

train['topics'] = train['Page content'].apply(extract_topics)
test['topics'] = test['Page content'].apply(extract_topics)

train['topics_clean'] = train['topics'].apply(clean_topics)
test['topics_clean'] = test['topics'].apply(clean_topics)


### (d) Final Tabular Features

- Combined all numerical attributes into a structured dataframe:

- `weekday`, `hour`, `days_since_start`, `channel_te`

In [9]:
def build_tabular_features(df):
    tabular = pd.DataFrame({
        "weekday": df['weekday'],
        "hour": df['hour'],
        "days_since_start": df['days_since_start'],
        "channel_te": df['channel_te'],
    })
    return tabular.fillna(0).astype(np.float32)

train_tab = build_tabular_features(train)
test_tab = build_tabular_features(test)


In [10]:
cv_topic = CountVectorizer(tokenizer=tokenizer_lemma_nostop, lowercase=False, min_df=2)
X_topic = cv_topic.fit_transform(train["topics_clean"])
X_topic_test = cv_topic.transform(test["topics_clean"])

print("Topic CountVec:", X_topic.shape)


Topic CountVec: (27643, 5609)


## 2. Classifier Construction

### 2.1 Model

- Used LightGBM (LGBMClassifier) for efficient handling of sparse and mixed-type features.

- Parameters:
    - `objective="binary"`, `n_estimators=300`, `learning_rate=0.01`, `n_jobs=1`

In [11]:
clf = lgb.LGBMClassifier(
    objective="binary",
    random_state=42,
    n_jobs=1,
    n_estimators=300,
    learning_rate=0.01
)

### 2.2 Feature Combination

- Concatenated:

    - Tabular numerical features (`csr_matrix(train_tab.values)`)

    - Topic bag-of-words features (`X_topic`)

In [12]:
X_lgb = hstack([csr_matrix(train_tab.values), X_topic])
X_lgb_test = hstack([csr_matrix(test_tab.values), X_topic_test])

print("Final train shape:", X_lgb.shape)
print("Final test shape:", X_lgb_test.shape)

Final train shape: (27643, 5613)
Final test shape: (11847, 5613)


### 2.3 Training & Evaluation

- Applied 5-fold cross-validation with ROC-AUC as the evaluation metric.

- Trained on full dataset after CV for final predictions.

In [13]:
clf.fit(X_lgb, y)

scores = cross_val_score(clf, X_lgb, y, cv=3, scoring="roc_auc", n_jobs=1)
print("LGBM CV AUC: %.5f (+/- %.5f)" % (scores.mean(), scores.std()))

clf.fit(X_lgb, y)
y_pred = clf.predict_proba(X_lgb_test)[:, 1]


[LightGBM] [Info] Number of positive: 13632, number of negative: 14011
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.054407 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 3298
[LightGBM] [Info] Number of data points in the train set: 27643, number of used features: 995
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.493145 -> initscore=-0.027423
[LightGBM] [Info] Start training from score -0.027423
[LightGBM] [Info] Number of positive: 9088, number of negative: 9340
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.023661 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2405
[LightGBM] [Info] Number of data points in the train set: 18428, number of used features: 683
[LightGBM] [Info] 

### Save Submission

In [14]:
submission = pd.DataFrame({
    "Id": test_raw["Id"],
    "Popularity": y_pred
})
submission.to_csv("submission_lgb.csv", index=False)
print("submission_lgb.csv saved!")

submission_lgb.csv saved!


## 3. Conclusions

### 3.1 Key Findings

- Temporal features (`weekday`, `hour`, `days_since_start`) captured posting-time behavior effectively.

- Channel **Target Encoding** significantly improved performance by modeling per-channel popularity tendencies.

- Topic-level bag-of-words provided complementary information to numeric patterns.

- The title text and content text don’t really matter.

### 3.2 Lessons Learned

- Preventing data leakage (especially from channel) is critical — improper handling can falsely boost AUC.

- Clean, interpretable preprocessing contributed more to performance than aggressive model tuning.

- LightGBM’s efficiency with sparse matrices made it ideal for text–tabular hybrid data.

### 3.3 Pitfalls

- Missing or malformed `<time>` tags occasionally caused NaT values; handled via `fillna(0)`.

### 3.4 Results
- **Final AUC (CV)**:  0.60532 (+/- 0.00373)
- **Public Score**: 0.60729
- **Private Score**: 0.59658